<a href="https://colab.research.google.com/github/EddyferO/Smart-Glass-Medical-Assistant/blob/main/notebooks/medgemma_in-context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load libraries

In [1]:
!pip install -q -U transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.7 MB/s eta 0:00:00


## Download data

In [2]:
!wget -nc -q "https://drive.google.com/uc?export=download&id=1w0_XjEl4gCrDWxFnsJHrhsVEGTiF3Ayh" -O trainingData.zip
!unzip -q trainingData.zip

## Collect drug data

In [3]:
import json
import glob
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Data Loading Logic
def load_drug_data(directory):
    all_data = []
    file_paths = glob.glob(os.path.join(directory, "*.json"))
    for path in file_paths:
        with open(path, 'r') as f:
            try:
                item = json.load(f)
                all_data.append(item)
            except: continue
    return all_data

drug_data = load_drug_data("/content/trainingFilesConverted/")

In [ ]:
drug_data[0]

## Get model and tokenizer

In [4]:
model_id = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

## Build in-context prompt

In [11]:
from IPython.display import display, Markdown

def run_in_context_extraction(target_drug, target_text, shots=2):
    # Select examples for the context
    context_examples = [ex for ex in drug_data if ex['drug'] != target_drug][:shots]

    # Build the prompt string
    prompt = "SYSTEM: You are a medical informatics expert. Extract drug-drug interactions into a structured JSON list.\n\n"

    for i, ex in enumerate(context_examples):
        prompt += f"### EXAMPLE {i+1}\n"
        prompt += f"DRUG: {ex['drug']}\n"
        prompt += f"TEXT: {ex['text'][:600]}...\n" # Truncated for token safety
        prompt += f"RESULT: {json.dumps(ex['interactions'])}\n\n"

    prompt += f"### TASK\nDRUG: {target_drug}\nTEXT:\n{target_text}\n\nRESULT:"

    # Visual
    display(Markdown("---"))
    display(Markdown(f"**The following text is being sent to the model:**\n\n```text\n{prompt}\n```"))

    # Execute
    formatted_input = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n[" # Force JSON start
    inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.01,
            do_sample=False
        )

    # Extract and format response
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    final_json = "[" + response

    display(Markdown(f"```json\n{final_json}\n```"))
    return final_json

## Sample it

In [12]:
# INPUT YOUR TEST CASE HERE
my_test_drug = "Warfarin" # @param {type: "string"}
my_test_text = "Warfarin metabolism is inhibited by fluconazole, leading to increased  prothrombin time. Concomitant administration of rifampin may decrease  warfarin effectiveness by inducing CYP enzymes. Patients should avoid  large amounts of leafy green vegetables (Vitamin K)." # @param {type: "string"}

# Execute
result = run_in_context_extraction(my_test_drug, my_test_text)

---

**The following text is being sent to the model:**

```text
SYSTEM: You are a medical informatics expert. Extract drug-drug interactions into a structured JSON list.

### EXAMPLE 1
DRUG: Ventavis
TEXT: 2. DOSAGE AND ADMINISTRATION 

 Ventavis is intended to be inhaled using the I-neb® AAD® System. Patients should receive 6 to 9 doses (inhalations) per day (minimum of 2 hours between doses during waking hours) as follows: 

 Starting dose: 2.5 mcg ( 2.1 ). 
 Uptitrate to 5 mcg if 2.5 mcg is well tolerated ( 2.1 ). 
 Maintenance dose: 5 mcg ( 2.1 ). 

 Delivered dose from ampule of : 

 Nebulizer 
 10 mcg/mL 
 20 mcg/mL 

 I-neb ® AAD ® 

 2.5 or 5 mcg from one ampule 
 5 mcg from one ampule 

 The 20 mcg/mL concentration is for patients who repeatedly experience extended treatment times ( 2.1...
RESULT: [{"@type": "Pharmacodynamic interaction", "@precipitant": "antihypertensive agents", "@precipitantCode": "N0000029427", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "vasodilators", "@precipitantCode": "N0000175940", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "anticoagulants", "@precipitantCode": "N0000029110", "@effect": "131148009: Bleeding (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "platelet inhibitors", "@precipitantCode": "N0000182142", "@effect": "131148009: Bleeding (finding)"}]

### EXAMPLE 2
DRUG: BRILINTA
TEXT: WARNING: (A) BLEEDING RISK, (B) ASPIRIN DOSE AND BRILINTA EFFECTIVENESS 

 A. BLEEDING RISK 


 BRILINTA, like other antiplatelet agents, can cause significant, sometimes fatal bleeding ( 5.1 , 6.1 ). 


 Do not use BRILINTA in patients with active pathological bleeding or a history of intracranial hemorrhage ( 4.1 , 4.2 ). 


 Do not start BRILINTA in patients undergoing urgent coronary artery bypass graft surgery (CABG) ( 5.1 , 6.1 ). 


 If possible, manage bleeding without discontinuing BRILINTA. Stopping BRILINTA increases the risk of subsequent cardiovascular events 

 (5.4) 

 . 

 B. A...
RESULT: [{"@type": "Pharmacodynamic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": "NO MAP"}, {"@type": "Unspecified interaction", "@precipitant": "p2y12 platelet inhibitor", "@precipitantCode": "N0000182142"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "lovastatin", "@precipitantCode": "N0000007106", "@effect": "NO MAP"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "lovastatin", "@precipitantCode": "N0000007106", "@effect": "C54357"}, {"@type": "Unspecified interaction", "@precipitant": "lovastatin", "@precipitantCode": "N0000007106"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "simvastatin", "@precipitantCode": "N0000005842", "@effect": "NO MAP"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "simvastatin", "@precipitantCode": "N0000005842", "@effect": "C54357"}, {"@type": "Unspecified interaction", "@precipitant": "simvastatin", "@precipitantCode": "N0000005842"}, {"@type": "Unspecified interaction", "@precipitant": "digoxin", "@precipitantCode": "N0000005903"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cyp3a inhibitors", "@precipitantCode": "N0000191001", "@effect": " 131148009: Bleeding (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cyp3a inhibitors", "@precipitantCode": "N0000191001", "@effect": "267036007: Dyspnea (finding)"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "cyp3a inhibitors", "@precipitantCode": "N0000191001", "@effect": "C54355"}, {"@type": "Unspecified interaction", "@precipitant": "cyp3a inhibitors", "@precipitantCode": "N0000191001"}, {"@type": "Unspecified interaction", "@precipitant": "atazanavir", "@precipitantCode": "N0000022320"}, {"@type": "Unspecified interaction", "@precipitant": "clarithromycin", "@precipitantCode": "N0000007316"}, {"@type": "Unspecified interaction", "@precipitant": "indinavir", "@precipitantCode": "N0000005722"}, {"@type": "Unspecified interaction", "@precipitant": "inhibitors of cyp3a", "@precipitantCode": "N0000191001"}, {"@type": "Unspecified interaction", "@precipitant": "itraconazole", "@precipitantCode": "N0000006753"}, {"@type": "Unspecified interaction", "@precipitant": "ketoconazole", "@precipitantCode": "N0000007319"}, {"@type": "Unspecified interaction", "@precipitant": "nefazodone", "@precipitantCode": "N0000007250"}, {"@type": "Unspecified interaction", "@precipitant": "nelfinavir", "@precipitantCode": "N0000006047"}, {"@type": "Unspecified interaction", "@precipitant": "ritonavir", "@precipitantCode": "N0000007423"}, {"@type": "Unspecified interaction", "@precipitant": "saquinavir", "@precipitantCode": "N0000007376"}, {"@type": "Unspecified interaction", "@precipitant": "telithromycin", "@precipitantCode": "N0000010551"}, {"@type": "Unspecified interaction", "@precipitant": "voriconazole", "@precipitantCode": "N0000010191"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "cyp3a inducers", "@precipitantCode": "n0000190118", "@effect": "C54356"}, {"@type": "Unspecified interaction", "@precipitant": "cyp3a inducers", "@precipitantCode": "n0000190118"}, {"@type": "Unspecified interaction", "@precipitant": "carbamazepine", "@precipitantCode": "N0000007470"}, {"@type": "Unspecified interaction", "@precipitant": "inducers of cyp3a", "@precipitantCode": "n0000190118"}, {"@type": "Unspecified interaction", "@precipitant": "phenobarbital", "@precipitantCode": "N0000005893"}, {"@type": "Unspecified interaction", "@precipitant": "phenytoin", "@precipitantCode": "N0000006023"}, {"@type": "Unspecified interaction", "@precipitant": "rifampin", "@precipitantCode": "N0000006026"}]

### TASK
DRUG: Warfarin
TEXT:
Warfarin metabolism is inhibited by fluconazole, leading to increased  prothrombin time. Concomitant administration of rifampin may decrease  warfarin effectiveness by inducing CYP enzymes. Patients should avoid  large amounts of leafy green vegetables (Vitamin K).

RESULT:
```

```json
[{"@type": "Pharmacokinetic interaction", "@precipitant": "fluconazole", "@precipitantCode": "N0000007316", "@effect": "131148009: Bleeding (finding)"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "rifampin", "@precipitantCode": "N0000006026", "@effect": "131148009: Bleeding (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "Vitamin K", "@precipitantCode": "N0000006023", "@effect": "131148009: Bleeding (finding)"}]

```

## API drug information testing

In [10]:
import requests
import json
from IPython.display import display, Markdown

def fetch_fda_interactions(drug_name):
    """
    Retrieves Section 7 (Drug Interactions) from the openFDA API.
    """
    # API endpoint
    base_url = "https://api.fda.gov/drug/label.json"

    # Build query
    params = {
        'search': f'openfda.brand_name:"{drug_name}" openfda.generic_name:"{drug_name}"',
        'limit': 1
    }

    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()

        if 'results' in data:
            label = data['results'][0]
            # 'drug_interactions' is the standard field for Section 7
            raw_text = label.get('drug_interactions', ["Section 7 not found in this label."])[0]

            return {
                "drug": drug_name.upper(),
                "text": raw_text,
                "source": "openFDA API"
            }
        return None
    except Exception as e:
        display(Markdown(f"**Error fetching {drug_name}:** {str(e)}"))
        return None

# Test
fetch_drug = "Epidiolex" # @param {type: "string"}
fetched = fetch_fda_interactions(fetch_drug)
if fetched:
    display(Markdown(f"### API Data: {fetched['drug']}"))
    display(Markdown(f"**Raw Section 7 Snippet:**\n> {fetched['text'][:500]}..."))

### API Data: EPIDIOLEX

**Raw Section 7 Snippet:**
> 7 DRUG INTERACTIONS • Strong inducer of CYP3A4 or CYP2C19: Consider dose increase of EPIDIOLEX. ( 7.1 ) • Consider a dose reduction of substrates of CYP1A2, CYP2C8, UGT1A9, and orally administered P-gp substrates. ( 7.2 ) • A lower starting dose of orally administered everolimus is recommended. ( 7.2 ) • Consider dose modification of CYP2B6 or CYP2C19 substrates. ( 7.2 ) 7.1 Effect of Other Drugs on EPIDIOLEX Strong CYP3A4 or CYP2C19 Inducers Concomitant use with a strong CYP3A4 and CYP2C19 indu...

In [9]:
def test_new_drug_with_medgemma(drug_name):
    # Collect the FDA label
    collected_data = fetch_fda_interactions(drug_name)

    if not collected_data:
        display(Markdown(f"Could not find a label for **{drug_name}**"))
        return

    # Pass the collected text to our extraction engine
    display(Markdown(f"###Testing on: **{drug_name}**"))

    final_json = run_in_context_extraction(
        target_drug=collected_data['drug'],
        target_text=collected_data['text']
    )

    return final_json

# Try a drug that wasn't in the training set to see if it understands the pattern
test_drug = "Amlodipine" # @param {type: "string"}
test_case = test_new_drug_with_medgemma(test_drug)

###Testing on: **Amlodipine**

---

**The following text is being sent to the model:**

```text
SYSTEM: You are a medical informatics expert. Extract drug-drug interactions into a structured JSON list.

### EXAMPLE 1
DRUG: Ventavis
TEXT: 2. DOSAGE AND ADMINISTRATION 

 Ventavis is intended to be inhaled using the I-neb® AAD® System. Patients should receive 6 to 9 doses (inhalations) per day (minimum of 2 hours between doses during waking hours) as follows: 

 Starting dose: 2.5 mcg ( 2.1 ). 
 Uptitrate to 5 mcg if 2.5 mcg is well tolerated ( 2.1 ). 
 Maintenance dose: 5 mcg ( 2.1 ). 

 Delivered dose from ampule of : 

 Nebulizer 
 10 mcg/mL 
 20 mcg/mL 

 I-neb ® AAD ® 

 2.5 or 5 mcg from one ampule 
 5 mcg from one ampule 

 The 20 mcg/mL concentration is for patients who repeatedly experience extended treatment times ( 2.1...
RESULT: [{"@type": "Pharmacodynamic interaction", "@precipitant": "antihypertensive agents", "@precipitantCode": "N0000029427", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "vasodilators", "@precipitantCode": "N0000175940", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "anticoagulants", "@precipitantCode": "N0000029110", "@effect": "131148009: Bleeding (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "platelet inhibitors", "@precipitantCode": "N0000182142", "@effect": "131148009: Bleeding (finding)"}]

### EXAMPLE 2
DRUG: BRILINTA
TEXT: WARNING: (A) BLEEDING RISK, (B) ASPIRIN DOSE AND BRILINTA EFFECTIVENESS 

 A. BLEEDING RISK 


 BRILINTA, like other antiplatelet agents, can cause significant, sometimes fatal bleeding ( 5.1 , 6.1 ). 


 Do not use BRILINTA in patients with active pathological bleeding or a history of intracranial hemorrhage ( 4.1 , 4.2 ). 


 Do not start BRILINTA in patients undergoing urgent coronary artery bypass graft surgery (CABG) ( 5.1 , 6.1 ). 


 If possible, manage bleeding without discontinuing BRILINTA. Stopping BRILINTA increases the risk of subsequent cardiovascular events 

 (5.4) 

 . 

 B. A...
RESULT: [{"@type": "Pharmacodynamic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": "NO MAP"}, {"@type": "Unspecified interaction", "@precipitant": "p2y12 platelet inhibitor", "@precipitantCode": "N0000182142"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "lovastatin", "@precipitantCode": "N0000007106", "@effect": "NO MAP"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "lovastatin", "@precipitantCode": "N0000007106", "@effect": "C54357"}, {"@type": "Unspecified interaction", "@precipitant": "lovastatin", "@precipitantCode": "N0000007106"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "simvastatin", "@precipitantCode": "N0000005842", "@effect": "NO MAP"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "simvastatin", "@precipitantCode": "N0000005842", "@effect": "C54357"}, {"@type": "Unspecified interaction", "@precipitant": "simvastatin", "@precipitantCode": "N0000005842"}, {"@type": "Unspecified interaction", "@precipitant": "digoxin", "@precipitantCode": "N0000005903"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cyp3a inhibitors", "@precipitantCode": "N0000191001", "@effect": " 131148009: Bleeding (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cyp3a inhibitors", "@precipitantCode": "N0000191001", "@effect": "267036007: Dyspnea (finding)"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "cyp3a inhibitors", "@precipitantCode": "N0000191001", "@effect": "C54355"}, {"@type": "Unspecified interaction", "@precipitant": "cyp3a inhibitors", "@precipitantCode": "N0000191001"}, {"@type": "Unspecified interaction", "@precipitant": "atazanavir", "@precipitantCode": "N0000022320"}, {"@type": "Unspecified interaction", "@precipitant": "clarithromycin", "@precipitantCode": "N0000007316"}, {"@type": "Unspecified interaction", "@precipitant": "indinavir", "@precipitantCode": "N0000005722"}, {"@type": "Unspecified interaction", "@precipitant": "inhibitors of cyp3a", "@precipitantCode": "N0000191001"}, {"@type": "Unspecified interaction", "@precipitant": "itraconazole", "@precipitantCode": "N0000006753"}, {"@type": "Unspecified interaction", "@precipitant": "ketoconazole", "@precipitantCode": "N0000007319"}, {"@type": "Unspecified interaction", "@precipitant": "nefazodone", "@precipitantCode": "N0000007250"}, {"@type": "Unspecified interaction", "@precipitant": "nelfinavir", "@precipitantCode": "N0000006047"}, {"@type": "Unspecified interaction", "@precipitant": "ritonavir", "@precipitantCode": "N0000007423"}, {"@type": "Unspecified interaction", "@precipitant": "saquinavir", "@precipitantCode": "N0000007376"}, {"@type": "Unspecified interaction", "@precipitant": "telithromycin", "@precipitantCode": "N0000010551"}, {"@type": "Unspecified interaction", "@precipitant": "voriconazole", "@precipitantCode": "N0000010191"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "cyp3a inducers", "@precipitantCode": "n0000190118", "@effect": "C54356"}, {"@type": "Unspecified interaction", "@precipitant": "cyp3a inducers", "@precipitantCode": "n0000190118"}, {"@type": "Unspecified interaction", "@precipitant": "carbamazepine", "@precipitantCode": "N0000007470"}, {"@type": "Unspecified interaction", "@precipitant": "inducers of cyp3a", "@precipitantCode": "n0000190118"}, {"@type": "Unspecified interaction", "@precipitant": "phenobarbital", "@precipitantCode": "N0000005893"}, {"@type": "Unspecified interaction", "@precipitant": "phenytoin", "@precipitantCode": "N0000006023"}, {"@type": "Unspecified interaction", "@precipitant": "rifampin", "@precipitantCode": "N0000006026"}]

### TASK
DRUG: AMLODIPINE
TEXT:
7 DRUG INTERACTIONS Do not exceed doses greater than 20 mg daily of simvastatin. ( 7.2 ) 7.1 Impact of Other Drugs on Amlodipine CYP3A Inhibitors Co-administration with CYP3A inhibitors (moderate and strong) results in increased systemic exposure to amlodipine and may require dose reduction. Monitor for symptoms of hypotension and edema when amlodipine is co-administered with CYP3A inhibitors to determine the need for dose adjustment [see CLINICAL PHARMACOLOGY ( 12.3 )] . CYP3A Inducers No information is available on the quantitative effects of CYP3A inducers on amlodipine. Blood pressure should be closely monitored when amlodipine is co-administered with CYP3A inducers. Sildenafil Monitor for hypotension when sildenafil is co-administered with amlodipine [see CLINICAL PHARMACOLOGY ( 12.2 )] . 7.2 Impact of Amlodipine on Other Drugs Simvastatin Co-administration of simvastatin with amlodipine increases the systemic exposure of simvastatin. Limit the dose of simvastatin in patients on amlodipine to 20 mg daily [see CLINICAL PHARMACOLOGY ( 12.3 )] . Immunosuppressants Amlodipine may increase the systemic exposure of cyclosporine or tacrolimus when co-administered. Frequent monitoring of trough blood levels of cyclosporine and tacrolimus is recommended and adjust the dose when appropriate [see CLINICAL PHARMACOLOGY ( 12.3 )] .

RESULT:
```

```json
[{"@type": "Pharmacokinetic interaction", "@precipitant": "CYP3A inhibitors", "@precipitantCode": "N0000191001", "@effect": "C54355"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "CYP3A inducers", "@precipitantCode": "n0000190118", "@effect": "C54356"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "sildenafil", "@precipitantCode": "N0000000000", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cyclosporine", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tacrolimus", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "simvastatin", "@precipitantCode": "N0000007106", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cyclosporine", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tacrolimus", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cyclosporine", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tacrolimus", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cyclosporine", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tacrolimus", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cyclosporine", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tacrolimus", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cyclosporine", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tacrolimus", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cyclosporine", "@precipitantCode": "N0000005903", "@effect": "131148009: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tac
```